# Artificial Intelligence — Lab 12
## Natural Language Processing and Computer Vision Mini Applications

**Course Learning Outcome — CLO6**  
Categorize and apply introductory concepts from advanced AI subdomains, including Natural Language Processing (NLP) and Computer Vision (CV).

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `matplotlib`, `scikit-learn`  
**Submission:** completed notebook containing predictions, code, experimental results, justifications, debugging answers, and reflection.

> **Assessment principle:** Correct code is only part of the evidence. Most marks come from your ability to **explain how raw text and images are represented numerically, justify preprocessing choices, interpret predictions, compare pipelines, and identify what belongs to NLP, CV, and Machine Learning**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. AI pipeline recap | 10 min | Distinguish raw data, representation, model, and output |
| 2. NLP mini-application | 40 min | Convert text to features and classify sentiment |
| 3. NLP lecture bridge | 10 min | Connect bag of words to embeddings and contextual representations |
| 4. Computer Vision mini-application | 30 min | Represent images numerically and classify digits |
| 5. Convolution lecture bridge | 15 min | Apply a simple local filter and interpret spatial features |
| 6. CV analysis & cross-domain comparison | 15 min | Interpret errors and compare NLP/CV pipelines |

> **Main idea:** NLP and Computer Vision work with very different raw inputs, but both require a suitable **representation** before a learning algorithm can use them. The practical methods here are intentionally simple and transparent bridges to the richer representations discussed in Lecture 6.

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish raw text from numerical text features;
2. explain a simple **bag-of-words** representation;
3. train and evaluate a basic text classifier;
4. identify limitations of bag-of-words and connect them to embeddings/contextual representations;
5. describe an image as a numeric array of pixel intensities;
6. explain why a 2D image may be flattened into a feature vector;
7. explain conceptually how a **local convolutional filter** preserves spatial structure better than flattening;
8. train and evaluate a simple image classifier;
9. compare NLP and Computer Vision pipelines;
10. distinguish the **AI application domain** from the **learning algorithm** used inside it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier

print("Lab 12 environment ready.")

# Part I — A General AI Pipeline

A simplified AI application often follows:

$$
\text{raw input}
\rightarrow
\text{representation}
\rightarrow
\text{model}
\rightarrow
\text{prediction}
$$

Examples:

### NLP

$$
\text{text}
\rightarrow
\text{word features}
\rightarrow
\text{classifier}
\rightarrow
\text{sentiment}
$$

### Computer Vision

$$
\text{image}
\rightarrow
\text{pixel features}
\rightarrow
\text{classifier}
\rightarrow
\text{digit class}
$$

## Task 1.1 — Representation Reasoning

Answer in your own words.

1. Why can a standard Machine Learning classifier not directly use an English sentence as a numeric feature vector?
2. Why is an image already closer to a numeric representation than natural-language text?
3. What does the **representation step** do?
4. Which parts of the pipeline are domain-specific, and which parts may be shared across AI domains?

**Your answers:**

# Part II — NLP Mini-Application: Sentiment Classification

We will classify short movie-review sentences as:

- `1` = positive;
- `0` = negative.

This is a deliberately small educational dataset. The purpose is to understand the NLP pipeline, not to build a production-quality sentiment system.

In [ ]:
texts = [
    "I loved this movie",
    "This film was excellent",
    "Amazing acting and wonderful story",
    "A fantastic and enjoyable experience",
    "The movie was brilliant",
    "I really liked the characters",
    "Great film with a strong ending",
    "The story was touching and beautiful",
    "Excellent direction and acting",
    "I would happily watch this again",
    "I hated this movie",
    "This film was terrible",
    "Awful acting and boring story",
    "A disappointing and unpleasant experience",
    "The movie was horrible",
    "I really disliked the characters",
    "Bad film with a weak ending",
    "The story was dull and predictable",
    "Poor direction and acting",
    "I would never watch this again",
]

labels = np.array([
    1,1,1,1,1,1,1,1,1,1,
    0,0,0,0,0,0,0,0,0,0
])

print("Number of texts:", len(texts))
print("Positive:", int(labels.sum()))
print("Negative:", int(len(labels) - labels.sum()))

## Task 2.1 — Identify the NLP Task

Answer:

1. What is the raw input?
2. What is the target label?
3. Is this classification or regression?
4. Is this supervised or unsupervised learning?
5. Which part makes this specifically an **NLP** task rather than a generic tabular ML task?

**Your answers:**

# Part III — Tokenization and Bag of Words

A very simple text representation is **bag of words**.

The idea:

1. build a vocabulary of known words;
2. count how often each vocabulary word occurs in each document;
3. represent every document by a numeric vector.

If the vocabulary is

```text
["good", "bad", "movie"]
```

then:

```text
"good movie good"
```

could become

$$
[2,0,1].
$$

Bag of words ignores word order.

## Task 3.1 — Manual Bag-of-Words Prediction

Suppose the vocabulary is:

```text
["great", "bad", "movie", "acting"]
```

Represent:

```text
"great movie great acting"
```

as a count vector.

Then represent:

```text
"bad acting"
```

Complete:

| Text | great | bad | movie | acting |
|---|---:|---:|---:|---:|
| `great movie great acting` |  |  |  |  |
| `bad acting` |  |  |  |  |

Then answer:

> What information is lost when we ignore word order?

**Your answer:**

In [ ]:
vectorizer = CountVectorizer(lowercase=True)

X_text = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X_text.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Some vocabulary terms:")
print(sorted(vectorizer.vocabulary_.keys())[:20])

## Task 3.2 — Interpret the Text Feature Matrix

If the feature matrix has shape

$$
(n_{\text{documents}}, n_{\text{vocabulary}})
$$

answer:

1. What does one row represent?
2. What does one column represent?
3. Why may the number of columns be much larger in a real NLP system?
4. Why is this representation usually sparse?

**Your answers:**

# Part IV — Train/Test Split for Text

We keep some sentences unseen during training.

In [ ]:
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    texts,
    labels,
    test_size=0.30,
    random_state=42,
    stratify=labels,
)

print("Training texts:", len(X_train_text))
print("Test texts:", len(X_test_text))

## Task 4.1 — Why Fit the Vectorizer on Training Text Only?

A common pipeline mistake is to let the representation step inspect all available data before evaluation.

Explain why the safer workflow is:

1. fit the vectorizer on **training text**;
2. transform training text;
3. transform test text using the same learned vocabulary.

**Your answer:**

In [ ]:
text_vectorizer = CountVectorizer(lowercase=True)

X_train_bow = text_vectorizer.fit_transform(X_train_text)
X_test_bow = text_vectorizer.transform(X_test_text)

print("Training BoW shape:", X_train_bow.shape)
print("Test BoW shape:", X_test_bow.shape)

# Part V — Train a Sentiment Classifier

We use Logistic Regression as a simple classifier.

The important point for this lab is not the internal mathematics of Logistic Regression, but the end-to-end NLP pipeline:

$$
\text{text}
\rightarrow
\text{vectorizer}
\rightarrow
\text{classifier}.
$$

In [ ]:
text_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

text_model.fit(X_train_bow, y_train_text)

text_predictions = text_model.predict(X_test_bow)

text_accuracy = accuracy_score(
    y_test_text,
    text_predictions
)

print("Text classification accuracy:", round(text_accuracy, 4))

## Task 5.1 — Predict and Interpret

Before inspecting individual outputs, answer:

1. Do you expect perfect test accuracy on such a small dataset?
2. Why can the exact result depend strongly on the train/test split?
3. Why should this lab result not be interpreted as a production-ready sentiment system?

**Your answer:**

In [ ]:
for text, actual, predicted in zip(
    X_test_text,
    y_test_text,
    text_predictions
):
    print(
        f"{text:<45} "
        f"| actual={actual} "
        f"| predicted={predicted} "
        f"| {'CORRECT' if actual == predicted else 'WRONG'}"
    )

## Task 5.2 — Analyze NLP Predictions

1. Identify one correctly classified sentence.
2. If there is an error, identify one misclassified sentence.
3. Which words in the sentence may have influenced the model?
4. Why is this explanation only approximate?
5. Why can words such as `"not"` be difficult for a simple bag-of-words representation?

**Your answers:**

# Part VI — Inspect Learned Word Weights

For binary Logistic Regression, positive coefficients push predictions toward class 1, while negative coefficients push predictions toward class 0.

In [ ]:
feature_names = np.array(
    text_vectorizer.get_feature_names_out()
)

weights = text_model.coef_[0]

top_positive_idx = np.argsort(weights)[-8:][::-1]
top_negative_idx = np.argsort(weights)[:8]

print("Words associated with positive sentiment:")
for i in top_positive_idx:
    print(feature_names[i], "->", round(weights[i], 3))

print("\nWords associated with negative sentiment:")
for i in top_negative_idx:
    print(feature_names[i], "->", round(weights[i], 3))

## Task 6.1 — Interpret the Word Weights

1. List two words strongly associated with positive sentiment.
2. List two words strongly associated with negative sentiment.
3. Do the learned associations make semantic sense?
4. Why can a word weight be unreliable when the training dataset is very small?
5. Does the model truly “understand” the sentence in the same way a human does?

**Your answers:**

# Part VII — Test New Sentences

Predict sentiment for new sentences not present in the original dataset.

In [ ]:
new_texts = [
    "The acting was excellent",
    "The movie was boring",
    "Wonderful story and great characters",
    "Poor film and terrible ending",
]

new_vectors = text_vectorizer.transform(new_texts)
new_predictions = text_model.predict(new_vectors)

for text, pred in zip(new_texts, new_predictions):
    print(
        f"{text:<40} -> "
        f"{'Positive' if pred == 1 else 'Negative'}"
    )

## Task 7.1 — NLP Generalization

Answer:

1. Which new predictions seem reasonable?
2. Which important limitation appears if a new sentence contains words never seen during training?
3. What does `CountVectorizer.transform()` do with unseen vocabulary?
4. Why would larger datasets and richer language representations improve real NLP systems?

**Your answers:**

## Lecture 6 Bridge — From Bag of Words to Richer NLP Representations

This lab intentionally uses **bag of words** because students can inspect every feature directly.

Lecture 6 also introduced richer representations:

- **embeddings** — dense learned vectors that can encode similarity;
- **contextual representations** — a word's representation can depend on surrounding words;
- **attention** — allows a model to weight relevant context differently.

Bag of words is therefore a useful baseline, not the final form of modern NLP.

### Task 7.2 — Connect the Practical Model to Lecture 6

Complete:

| Representation idea | What it can represent better than simple counts |
|---|---|
| Bag of words |  |
| Embeddings |  |
| Contextual representations / attention |  |

Then answer:

1. Which representation in this lab ignores word order?
2. Why can embeddings help represent semantic similarity?
3. Why might the word `"bank"` need different contextual representations in `"river bank"` and `"bank account"`?
4. Why are we **not** implementing a transformer in this introductory lab?

**Your answers:**

# Part VIII — NLP Debugging

## Task 8.1 — Different Vocabulary for Test Data

A student writes:

```python
train_vectorizer.fit_transform(train_text)
test_vectorizer.fit_transform(test_text)
```

using two separate vectorizers.

1. Why can the feature columns no longer be guaranteed to mean the same thing?
2. Why is this incompatible with the trained classifier?
3. What should be done instead?

**Your answer:**

## Task 8.2 — Bag of Words and Negation

Consider:

```text
"I like this movie"
"I do not like this movie"
```

Explain why a very simple bag-of-words model may struggle to represent the semantic difference correctly.

**Your answer:**

# Part IX — Computer Vision Mini-Application: Handwritten Digits

We now switch from text to images.

We use the built-in `digits` dataset from `scikit-learn`.

Each image represents one handwritten digit from 0 to 9.

In [ ]:
digits = load_digits()

X_img = digits.data
y_img = digits.target
images = digits.images

print("Flattened feature matrix:", X_img.shape)
print("Image array shape:", images.shape)
print("Number of classes:", len(np.unique(y_img)))

## Task 9.1 — Interpret the Image Shapes

The digits images are $8\times8$ grayscale images.

Answer:

1. How many pixels are in one image?
2. Why does `digits.data` have 64 features per example?
3. What does one pixel value represent?
4. Why can an image be treated as numerical data?

**Your answers:**

In [ ]:
for i in range(5):
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(images[i])
    plt.title(f"Label: {y_img[i]}")
    plt.axis("off")
    plt.show()

## Task 9.2 — Visual Reasoning

Inspect the displayed digits.

1. What visual information allows a human to distinguish the classes?
2. Why may two examples of the same digit look different?
3. Why can two different digits sometimes look similar?
4. Why does this make image classification a nontrivial AI problem?

**Your answers:**

# Part X — Flattening an Image

The original image has shape

$$
8\times8.
$$

A simple classifier may receive it as a vector of length

$$
64.
$$

Flattening changes the representation:

$$
\text{2D pixel grid}
\rightarrow
\text{1D feature vector}.
$$

It does **not** change the underlying pixel values.

In [ ]:
example_image = images[0]
example_vector = X_img[0]

print("Image shape:", example_image.shape)
print("Flattened vector shape:", example_vector.shape)

print("\nFirst image:")
print(example_image)

print("\nFirst 16 flattened values:")
print(example_vector[:16])

## Task 10.1 — Explain Flattening

1. Why does flattening produce 64 features?
2. What spatial information is harder to represent explicitly after flattening?
3. Why do modern Computer Vision systems often use architectures that preserve spatial structure better than a simple flat vector?

**Your answers:**

## Lecture 6 Bridge — Convolution and Local Spatial Features

Flattening converts an $8\times8$ image into 64 numbers, but it hides the image's explicit 2D neighborhood structure.

Lecture 6 introduced **convolution** as a way to process local image regions.

A small filter (kernel) is moved across an image. At each location, the filter combines nearby pixel values to produce a **feature response**.

Consider the vertical-edge filter:

$$
K=
\begin{bmatrix}
-1 & 0 & 1\\
-1 & 0 & 1\\
-1 & 0 & 1
\end{bmatrix}.
$$

A strong response suggests a local left-to-right intensity change.

### Task 10.2 — Manual Local Filter Calculation

For the patch

$$
P=
\begin{bmatrix}
0 & 0 & 1\\
0 & 0 & 1\\
0 & 0 & 1
\end{bmatrix},
$$

compute the filter response

$$
\sum_{i,j} P_{ij}K_{ij}.
$$

Then answer:

1. Is the response positive, negative, or zero?
2. What local visual pattern does the kernel try to detect?
3. Why is this operation more spatially meaningful than treating all 64 pixels as unrelated features?

**Your answer:**

In [ ]:
vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

def apply_valid_filter(image, kernel):
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)

    kh, kw = kernel.shape
    ih, iw = image.shape

    out_h = ih - kh + 1
    out_w = iw - kw + 1

    response = np.zeros((out_h, out_w), dtype=float)

    for r in range(out_h):
        for c in range(out_w):
            patch = image[r:r+kh, c:c+kw]
            response[r, c] = np.sum(patch * kernel)

    return response

example_digit = images[0]
edge_response = apply_valid_filter(
    example_digit,
    vertical_edge_kernel
)

print("Original image shape:", example_digit.shape)
print("Filter response shape:", edge_response.shape)
print("Maximum response:", round(float(edge_response.max()), 3))
print("Minimum response:", round(float(edge_response.min()), 3))

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(example_digit)
plt.title("Original Digit Image")
plt.axis("off")
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(edge_response)
plt.title("Vertical-Edge Filter Response")
plt.axis("off")
plt.show()

### Task 10.3 — Interpret the Convolution Bridge

1. Why is the filter output $6\times6$ rather than $8\times8$ in this **valid** convolution?
2. Where do you see strong positive or negative responses?
3. What kind of local image structure causes these responses?
4. How does this connect to Lecture 6's idea that early CNN layers can detect edges and simple patterns?
5. Why is this still only a **conceptual bridge**, not a full CNN implementation?

**Your answers:**

# Part XI — Train/Test Split for Digits

In [ ]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img,
    y_img,
    test_size=0.20,
    random_state=42,
    stratify=y_img,
)

print("Training images:", len(X_train_img))
print("Test images:", len(X_test_img))

## Task 11.1 — Prediction Before Training

We will use a $k$-Nearest Neighbors classifier with:

```text
k = 3
```

Before running:

1. What does KNN compare a new image with?
2. Why should images of the same digit often be close in feature space?
3. Why might handwriting variation still cause classification errors?

**Your prediction:**

In [ ]:
image_model = KNeighborsClassifier(
    n_neighbors=3
)

image_model.fit(
    X_train_img,
    y_train_img
)

image_predictions = image_model.predict(
    X_test_img
)

image_accuracy = accuracy_score(
    y_test_img,
    image_predictions
)

print("Digit classification accuracy:", round(image_accuracy, 4))

## Task 11.2 — Interpret CV Accuracy

1. What is the test accuracy?
2. Approximately how many test images were classified correctly?
3. Why should the model be evaluated on unseen images?
4. Why does high accuracy not mean every individual prediction is correct?

**Your answers:**

# Part XII — Inspect Computer Vision Errors

In [ ]:
error_indices = np.where(
    image_predictions != y_test_img
)[0]

print("Number of misclassified test images:", len(error_indices))

n_show = min(6, len(error_indices))

if n_show > 0:
    for idx in error_indices[:n_show]:
        image = X_test_img[idx].reshape(8, 8)

        plt.figure(figsize=(2.5, 2.5))
        plt.imshow(image)
        plt.title(
            f"Actual {y_test_img[idx]} | "
            f"Pred {image_predictions[idx]}"
        )
        plt.axis("off")
        plt.show()
else:
    print("No errors to display.")

## Task 12.1 — Analyze an Image Error

Choose one misclassified image, if available.

1. What is the actual digit?
2. What did the model predict?
3. Does the image look ambiguous to you?
4. What visual similarity may have caused the confusion?
5. Why is inspecting errors useful beyond reporting one accuracy number?

**Your answers:**

# Part XIII — Confusion Matrix for Digits

In [ ]:
cm_img = confusion_matrix(
    y_test_img,
    image_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_img
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title("Digit Classification Confusion Matrix")
plt.show()

## Task 13.1 — Interpret the Digit Confusion Matrix

1. Which diagonal entries correspond to correct predictions?
2. Identify one pair of digits that is confused, if any.
3. Why can a confusion matrix help identify specific weaknesses of a vision model?
4. How is this interpretation similar to the sentiment-classification confusion matrix from Lab 11?

**Your answers:**

# Part XIV — Personalized KNN Variation

Use the last digit of your student ID.

- `0–3`: $k=1$
- `4–6`: $k=3$
- `7–9`: $k=7$

Use the same train/test split.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_K = 1
    elif 4 <= LAST_DIGIT <= 6:
        PERSONAL_K = 3
    elif 7 <= LAST_DIGIT <= 9:
        PERSONAL_K = 7
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned k:", PERSONAL_K)

## Task 14.1 — Predict Before Running

Write:

- **My assigned $k$:**  
- **Do I expect more local/sensitive or smoother decisions?**  
- **Do I expect test accuracy to increase, decrease, or remain similar?**  
- **Why?**

Then run your assigned model.

In [ ]:
if LAST_DIGIT is not None:
    personal_knn = KNeighborsClassifier(
        n_neighbors=PERSONAL_K
    )

    personal_knn.fit(
        X_train_img,
        y_train_img
    )

    personal_predictions = personal_knn.predict(
        X_test_img
    )

    personal_accuracy = accuracy_score(
        y_test_img,
        personal_predictions
    )

    print(
        "Personalized digit accuracy:",
        round(personal_accuracy, 4)
    )

## Task 14.2 — Explain the Personalized Result

1. Was your accuracy prediction correct?
2. What changed in the learning algorithm?
3. What did **not** change in the Computer Vision problem?
4. Why is choosing $k$ a model-design decision rather than a change to the image domain itself?

**Your answers:**

# Part XV — Compare NLP and Computer Vision

Complete the table.

| Pipeline Component | NLP Mini-App | CV Mini-App |
|---|---|---|
| Raw input |  |  |
| Simple representation used in lab |  |  |
| Richer representation discussed in Lecture 6 |  |  |
| Learning algorithm used in lab |  |  |
| Output |  |  |
| Evaluation metric |  |  |

Then answer:

1. What is common between the two pipelines?
2. What is domain-specific?
3. Why is Logistic Regression not itself “NLP”?
4. Why is KNN not itself “Computer Vision”?
5. How do **embeddings/attention** and **convolution** illustrate the importance of representation in different domains?

**Your answers:**

# Part XVI — Debugging Across Domains

## Task 16.1 — Text Representation Mismatch

A text classifier is trained using one vocabulary, but test sentences are transformed using a different independently learned vocabulary.

Why is this invalid?

**Your answer:**

## Task 16.2 — Image Shape Mismatch

A classifier was trained on flattened vectors of length 64, but a student passes a raw $8\times8$ matrix directly to `.predict()`.

Why can this fail?

**Your answer:**

## Task 16.3 — Domain vs. Algorithm Confusion

A student says:

> “Logistic Regression is NLP and KNN is Computer Vision.”

Explain why this statement is conceptually incorrect.

**Your answer:**

# Part XVII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where text becomes a numeric vector.
- What does one bag-of-words column represent?
- Why can negation be difficult for bag of words?
- How are embeddings conceptually different from word counts?
- Why can context change the representation of a word?
- Show me where an image is represented by 64 features.
- What information does one pixel contain?
- What does the simple convolution filter detect?
- Why does convolution preserve local spatial structure better than flattening?
- Why is Logistic Regression not itself an NLP method?
- Why is KNN not itself a Computer Vision method?
- What changed in your personalized $k$ experiment?

> You are expected to explain the **AI concept represented by the code**, not memorize library syntax.

# Reflection

Answer concisely but precisely.

### R1 — Representation
Why is representation one of the most important differences between AI domains?

**Answer:**

### R2 — NLP
Why is bag of words useful as a teaching model, and what important information does it lose?

**Answer:**

### R3 — Richer NLP
How do embeddings and contextual representations address limitations of simple word counts?

**Answer:**

### R4 — Computer Vision
Why can flattened pixel vectors work for simple digit classification, while convolution is better suited to exploiting local spatial structure?

**Answer:**

### R5 — AI Subdomains
Explain the difference between:
- an **application domain** such as NLP or Computer Vision;
- a **representation** such as bag of words, embeddings, pixels, or feature maps;
- a **learning algorithm** such as Logistic Regression or KNN.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] AI-pipeline representation reasoning;
- [ ] NLP task identification;
- [ ] manual bag-of-words calculation;
- [ ] text feature-matrix interpretation;
- [ ] train/test vectorizer justification;
- [ ] sentiment classifier results;
- [ ] NLP prediction/error analysis;
- [ ] learned word-weight interpretation;
- [ ] lecture bridge from bag of words to embeddings/contextual representations;
- [ ] NLP debugging answers;
- [ ] image-shape and pixel interpretation;
- [ ] flattening explanation;
- [ ] manual convolution/filter calculation;
- [ ] interpretation of the local edge-filter response;
- [ ] digit classifier results;
- [ ] CV error analysis;
- [ ] digit confusion-matrix interpretation;
- [ ] personalized KNN experiment;
- [ ] cross-domain comparison table;
- [ ] debugging answers;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab12_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Valid NLP/CV pipelines, predictions, evaluation, and filter experiment |
| **Conceptual / algorithmic justification** | **3.0** | Explains representations, bag-of-words limitations, embeddings/context, pixels, convolution, and model/domain distinction |
| **Experimental analysis** | **2.0** | Interprets NLP/CV predictions, errors, confusion matrices, and filter responses |
| **Prediction / debugging / reasoning** | **1.0** | Manual representation/filter tasks, predictions, and diagnosis of pipeline mistakes |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct predictions without an adequate explanation of how text/images were represented and processed earn only limited credit.

## Key Takeaways

- NLP and Computer Vision use different raw data but share a common AI pipeline.
- Text must be transformed into numeric features before standard classifiers can use it.
- Bag of words is simple and transparent, but ignores order and context.
- Embeddings and contextual representations provide richer ways to represent language.
- Images can be represented numerically through pixel intensities.
- Flattened image vectors can work for simple tasks but hide explicit spatial organization.
- Convolution applies local filters and preserves neighborhood structure, providing a bridge to the CNN concepts discussed in Lecture 6.
- Classification accuracy should be supplemented with error inspection and confusion matrices.
- The **application domain**, the **representation**, and the **learning algorithm** are different concepts.

This lab completes the core practical sequence across **search, optimization, CSPs, adversarial search, Machine Learning, NLP, and Computer Vision**.